# Day 26 — SQL Project: Month-End Close Queries
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Build production-quality SQL queries for a month-end close reporting package
2. Implement flash report logic: current month vs prior month vs prior year same month
3. Detect accrual anomalies using SQL threshold logic

---
> **SAP Context:** Month-end close queries map directly to Financial Accounting (FI) and Controlling (CO) reports. In SAP, FBL3N gives GL line items, KSB1 gives CO actuals, S_ALR_87013611 gives variance reports. These SQL queries replicate the aggregation layer that BW or SAP Analytics Cloud would build on top of similar extracts.


## 0. Setup — Load All Datasets into SQLite

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
# In-memory DuckDB — register CSVs as tables/views
con = duckdb.connect()

tables = {
    'materials':    'materials_inventory.csv',
    'sales_orders': 'sales_orders.csv',
    'cost_centers': 'cost_center_actuals.csv',
    'hr':           'hr_headcount.csv',
    'bw_kpis':      'bw_sales_kpis.csv',
}

dfs = {}
for tname, fname in tables.items():
    df = pd.read_csv(DATA / fname)
    con.register(tname, df)
    dfs[tname] = df

def q(sql):
    return con.sql(sql).df()

for t in tables:
    print(f'Loaded {t}: {len(dfs[t])} rows')

print('\nAll tables loaded.')


## 1. P&L Summary — Revenue, COGS, Gross Margin by Period and Division

In [ ]:
# YOUR SOLUTION
# Business context: This is the top-line P&L view a CFO would see on Day 1 of close.
# Revenue and COGS come from the BW extract; SPART = product division.

sql_pl = '''
SELECT
    CAL_YEAR_MONTH                          AS Period,
    SPART                                   AS Division,
    ROUND(SUM(REVENUE), 2)                  AS Revenue,
    ROUND(SUM(COGS), 2)                     AS COGS,
    ROUND(SUM(GROSS_MARGIN), 2)             AS Gross_Margin,
    ROUND(SUM(GROSS_MARGIN)/NULLIF(SUM(REVENUE),0)*100, 1) AS GM_Pct
FROM bw_kpis
GROUP BY CAL_YEAR_MONTH, SPART
ORDER BY CAL_YEAR_MONTH, SPART
'''

pl_df = q(sql_pl)
print('=== P&L Summary (sample) ===')
print(pl_df.head(20).to_string(index=False))

## 2. Top Variance Drivers — Cost Elements with Largest Plan vs Actual Gap

In [ ]:
# YOUR SOLUTION
# Business context: On Day 2 of close, the Controller wants to know
# which cost elements explain the budget variance. This is your KSB1 drill-down.

sql_variance = '''
SELECT
    KOSTL                                   AS Cost_Center,
    KTEXT                                   AS CC_Description,
    KSTAR                                   AS Cost_Element,
    KSTAR_DESC                              AS CE_Description,
    GJAHR                                   AS Fiscal_Year,
    SUM(ACTUAL_AMT)                         AS Actual,
    SUM(PLAN_AMT)                           AS Plan,
    ROUND(SUM(ACTUAL_AMT) - SUM(PLAN_AMT), 2) AS Variance,
    ROUND(
        (SUM(ACTUAL_AMT) - SUM(PLAN_AMT)) / NULLIF(SUM(PLAN_AMT), 0) * 100,
        1
    )                                       AS Variance_Pct
FROM cost_centers
WHERE GJAHR = 2024
GROUP BY KOSTL, KTEXT, KSTAR, KSTAR_DESC, GJAHR
ORDER BY ABS(SUM(ACTUAL_AMT) - SUM(PLAN_AMT)) DESC
LIMIT 20
'''

var_df = q(sql_variance)
print('=== Top 20 Variance Drivers (FY2024) ===')
print(var_df.to_string(index=False))

## 3. Flash Report — Current Month vs Prior Month vs Prior Year Same Month

In [ ]:
# YOUR SOLUTION
# Business context: The flash report is sent on Day 3 of close to senior leadership.
# It shows three comparative columns for revenue performance.

# Use 202412 as 'current month' for this exercise
CURRENT_MONTH = 202412
PRIOR_MONTH = 202411
PRIOR_YEAR_SAME = 202312

sql_flash = f'''
SELECT
    'Revenue'       AS KPI,
    ROUND(SUM(CASE WHEN CAL_YEAR_MONTH = {CURRENT_MONTH} THEN REVENUE ELSE 0 END), 0) AS Current_Month,
    ROUND(SUM(CASE WHEN CAL_YEAR_MONTH = {PRIOR_MONTH}   THEN REVENUE ELSE 0 END), 0) AS Prior_Month,
    ROUND(SUM(CASE WHEN CAL_YEAR_MONTH = {PRIOR_YEAR_SAME} THEN REVENUE ELSE 0 END), 0) AS Prior_Year_Same_Month
FROM bw_kpis

UNION ALL

SELECT
    'Gross Margin $' AS KPI,
    ROUND(SUM(CASE WHEN CAL_YEAR_MONTH = {CURRENT_MONTH} THEN GROSS_MARGIN ELSE 0 END), 0),
    ROUND(SUM(CASE WHEN CAL_YEAR_MONTH = {PRIOR_MONTH}   THEN GROSS_MARGIN ELSE 0 END), 0),
    ROUND(SUM(CASE WHEN CAL_YEAR_MONTH = {PRIOR_YEAR_SAME} THEN GROSS_MARGIN ELSE 0 END), 0)
FROM bw_kpis

UNION ALL

SELECT
    'Orders' AS KPI,
    SUM(CASE WHEN CAL_YEAR_MONTH = {CURRENT_MONTH} THEN NUM_ORDERS ELSE 0 END),
    SUM(CASE WHEN CAL_YEAR_MONTH = {PRIOR_MONTH}   THEN NUM_ORDERS ELSE 0 END),
    SUM(CASE WHEN CAL_YEAR_MONTH = {PRIOR_YEAR_SAME} THEN NUM_ORDERS ELSE 0 END)
FROM bw_kpis
'''

flash_df = q(sql_flash)
print('=== Flash Report ===')
print(flash_df.to_string(index=False))

## 4. Accrual Check — Detect Unusually Low Actuals (Possible Missing Postings)

In [ ]:
# YOUR SOLUTION
# Business context: At month-end close, controllers check for periods where actuals
# are suspiciously low — often a sign that a recurring journal entry wasn't posted.
# Rule: flag any cost center/period where actual < 50% of the average for that cost center.

sql_accrual = '''
WITH avg_actual AS (
    SELECT
        KOSTL,
        KSTAR,
        AVG(ACTUAL_AMT) AS avg_amt,
        STDDEV(ACTUAL_AMT) AS stddev_amt   -- SQLite doesn't have STDDEV; use manual calc
    FROM cost_centers
    WHERE GJAHR = 2024
    GROUP BY KOSTL, KSTAR
),
-- Recalculate without STDDEV (SQLite compatible)
cc_avg AS (
    SELECT
        KOSTL,
        KSTAR,
        AVG(ACTUAL_AMT) AS avg_amt
    FROM cost_centers
    WHERE GJAHR = 2024
    GROUP BY KOSTL, KSTAR
)
SELECT
    c.KOSTL,
    c.KTEXT,
    c.KSTAR,
    c.KSTAR_DESC,
    c.GJAHR,
    c.PERIOD,
    ROUND(c.ACTUAL_AMT, 2) AS Actual,
    ROUND(a.avg_amt, 2) AS Avg_Actual,
    ROUND(c.ACTUAL_AMT / NULLIF(a.avg_amt, 0) * 100, 1) AS Pct_Of_Avg,
    'CHECK ACCRUAL' AS Flag
FROM cost_centers c
JOIN cc_avg a ON c.KOSTL = a.KOSTL AND c.KSTAR = a.KSTAR
WHERE c.GJAHR = 2024
  AND c.ACTUAL_AMT > 0
  AND c.ACTUAL_AMT < a.avg_amt * 0.50   -- less than 50% of average = suspect
ORDER BY Pct_Of_Avg ASC
LIMIT 20
'''

accrual_df = q(sql_accrual)
print(f'=== Potential Missing Accruals: {len(accrual_df)} records ===')
print(accrual_df.to_string(index=False))

## Daily Prompt
**Answer independently:**

1. The Controller notices the flash report shows December 2024 revenue is 0. Debug this: what would you check first in the data? Write the SQL to diagnose.
2. Modify the accrual check query to also flag periods where actuals are MORE than 150% of average (potential double-posting). Combine both flags into a single result set with a "FLAG_TYPE" column ('UNDER' or 'OVER').
3. In SAP FI, what is the difference between a "parked" document and a "posted" document? Why does this matter for close queries?
